# Load Env

In [1]:
from pathlib import Path
import sys
import logging

project_root = str(Path("/home/user/perso/trading/alphalab").resolve())

if project_root not in sys.path:
    sys.path.append(project_root)

# Configuration de l'autocomplétion Jupyter
%config IPCompleter.use_jedi = False
%config IPCompleter.greedy = False

# Activation de l'autoreload
%load_ext autoreload
%autoreload 2

logging.basicConfig(
    level=logging.INFO,
    format='[%(levelname)s] %(message)s',
    stream=sys.stdout,
    force=False,
)

logging.getLogger('enl').setLevel(logging.INFO)

# Atelier

## Test fit()

In [5]:
import numpy as np
import pytest
from enl.linear_model import LinearFactorModel

def test_linear_factor_model_fit():
    # Cas nominal et validation de la colinéarité
    Y_nominal = np.array([[1.10], [1.20]])
    X_nominal = np.array([[1.25], [1.30]])
    model = LinearFactorModel()
    beta = model.fit(X_nominal, Y_nominal)

    assert model.X_design.shape == (2, 2)
    assert beta.shape == (2, 1)
    np.testing.assert_allclose(beta, [[-1.40], [2.00]], rtol=1e-5, atol=1e-5)

    X_colineaire = np.array([[1.25, 2.50], [1.28, 2.56], [1.22, 2.44]])
    Y_colineaire = np.array([[1.10], [1.15], [1.08]])
    model_secure = LinearFactorModel()

    with pytest.raises(ValueError):
        model_secure.fit(X_colineaire, Y_colineaire)
    print("✅ TOUS LES TESTS UNITAIRES ONT REUSSI AVEC SUCCES !")

test_linear_factor_model_fit()


✅ TOUS LES TESTS UNITAIRES ONT REUSSI AVEC SUCCES !


# Residual covariance

In [6]:
def test_residual_covariance():
    # On simule un modèle où les résidus sont connus (ex: e1 = 0.5, e2 = 1.0)
    model = LinearFactorModel()
    model.residuals = np.array([[0.5], [1.0]])

    omega = model.get_residual_covariance()

    # On attend la matrice carrée 2x2 des produits croisés
    attendu_omega = np.array([[0.25, 0.5],
                              [0.5,  1.0]])

    assert omega.shape == (2, 2)
    np.testing.assert_allclose(omega, attendu_omega, rtol=1e-5)
    print("✅ ÉTAPE GET_RESIDUAL_COVARIANCE VALIDÉE AVEC SUCCÈS !")

test_residual_covariance()

✅ ÉTAPE GET_RESIDUAL_COVARIANCE VALIDÉE AVEC SUCCÈS !


# Bias hypothesis

In [14]:
from enl.linear_model import *

def test_check_bias_binary_pure_unit():
    model = LinearFactorModel()

    # CAS 1 : Modèle non biaisé (Moyenne = 0.0)
    model.residuals = np.array([[-1.0], [1.0]])
    # np.mean est de 0.0, inférieur au seuil de 1e-5
    assert model.check_bias_hypothesis(threshold=1e-5) is True
    print("✅ Sous-test 1 (Non biaisé) : Validé ! Le modèle donne le feu vert (True).")

    # CAS 2 : Modèle biaisé (Erreur systématique positive)
    model.residuals = np.array([[0.5], [1.5]])
    # np.mean est de 1.0, ce qui crève le plafond du seuil de 1e-5
    assert model.check_bias_hypothesis(threshold=1e-5) is False
    print("✅ Sous-test 2 (Biaisé) : Validé ! Le coupe-circuit s'est activé (False).")

test_check_bias_binary_pure_unit()

✅ Sous-test 1 (Non biaisé) : Validé ! Le modèle donne le feu vert (True).
✅ Sous-test 2 (Biaisé) : Validé ! Le coupe-circuit s'est activé (False).


# Auto-corr

In [15]:
def test_check_autocorrelation_pure_unit():
    model = LinearFactorModel()

    # CAS 1 : Pas d'autocorrélation (Covariance nulle)
    # res_t   = [ 1.0, -1.0]
    # res_lag = [-1.0,  1.0]
    # Produit croisé : (1.0 * -1.0) + (-1.0 * 1.0) = -1.0 + -1.0 = -2.0 ?
    # Prenons un cas orthogonal parfait :
    model.residuals = np.array([[1.0], [0.0], [-1.0], [0.0]])
    # res_t   = [[0.0], [-1.0], [0.0]]
    # res_lag = [[1.0], [0.0], [-1.0]]
    # Produit croisé : (0*1) + (-1*0) + (0*-1) = 0.0
    assert model.check_autocorrelation_hypothesis(covariance_threshold=1e-5) is True
    print("✅ Sous-test 1 (Indépendant) : Validé ! Aucun motif temporel détecté (True).")

    # CAS 2 : Forte autocorrélation positive (Inertie du spread)
    # Les résidus gardent le même signe
    model.residuals = np.array([[1.0], [2.0], [3.0]])
    # res_t   = [[2.0], [3.0]]
    # res_lag = [[1.0], [2.0]]
    # Covariance = (2.0 * 1.0) + (3.0 * 2.0) = 2.0 + 6.0 = 8.0
    assert model.check_autocorrelation_hypothesis(covariance_threshold=1e-5) is False
    print("✅ Sous-test 2 (Mémoire) : Validé ! Autocorrélation détectée, coupe-circuit activé (False).")

test_check_autocorrelation_pure_unit()

✅ Sous-test 1 (Indépendant) : Validé ! Aucun motif temporel détecté (True).
✅ Sous-test 2 (Mémoire) : Validé ! Autocorrélation détectée, coupe-circuit activé (False).


# Multi-collinearity

In [16]:
import numpy as np

def test_check_multicollinearity_pure_unit():
    model = LinearFactorModel()

    # CAS 1 : Facteurs sains et indépendants (Matrice X bien proportionnée)
    model.X_design = np.array([[1.0, 1.0, 0.0],
                               [1.0, 0.0, 1.0],
                               [1.0, -1.0, -1.0]])
    # Le nombre de conditionnement sera bas, bien en dessous de 30
    assert model.check_multicollinearity_hypothesis(max_condition_number=30.0) is True
    print("✅ Sous-test 1 (Facteurs sains) : Validé ! Le modèle est jugé stable (True).")

    # CAS 2 : Quasi-colinéarité (La colonne 3 est presque égale à la colonne 2)
    model.X_design = np.array([[1.0, 1.0, 1.00001],
                               [1.0, 2.0, 2.00002],
                               [1.0, 3.0, 3.00001]])
    # Les variables se marchent dessus, le nombre de conditionnement explose
    assert model.check_multicollinearity_hypothesis(max_condition_number=30.0) is False
    print("✅ Sous-test 2 (Quasi-colinéarité) : Validé ! Instabilité détectée, sécurité activée (False).")

test_check_multicollinearity_pure_unit()

✅ Sous-test 1 (Facteurs sains) : Validé ! Le modèle est jugé stable (True).
✅ Sous-test 2 (Quasi-colinéarité) : Validé ! Instabilité détectée, sécurité activée (False).
